# Local Agentic AI with Apple MLX & Gemma 4 (12B) — Lab 1 Guide

This notebook demonstrates setting up and running **Google Gemma 4 (12B)** locally on Apple Silicon using **Apple MLX** (`mlx_vlm`) — an array framework purpose-built for high-performance machine learning on Apple Silicon Metal GPU and Unified Memory. This forms the foundational local LLM layer for autonomous AI agents with zero cloud subscription costs, no rate limits, and 100% data privacy.

| Model Variant | Parameters | Precision / Quant | Local Size | VRAM / Unified RAM | Best Use Case |
|---|---|---|---|---|---|
| `gemma-4-e2b` | 2.1 B | 4-bit | ~1.6 GB | ~2.5 GB | Ultra-fast edge devices, mobile, IoT |
| `gemma-4-e4b` | 4.3 B | 4-bit | ~3.3 GB | ~4.5 GB | Fast conversational tasks |
| `gemma-4-12B-it-qat-4bit` | 12.1 B | 4-bit QAT | ~10.0 GB | ~11.5 GB | ✅ **Recommended local sweet spot** for complex agentic reasoning & code |
| `gemma-4-27b` | 27.2 B | 4-bit | ~17.0 GB | ~20.0 GB | Frontier open-weights desktop deployment |

> **Why Local MLX for Agents?** Autonomous agents execute dozens or hundreds of prompt-and-response loops for chain-of-thought reasoning, self-reflection, and tool invocation. Running `gemma-4-12B` on Apple MLX utilizes zero-copy unified memory, eliminating PCI transfer bottlenecks, eliminating per-token API fees, and providing sub-millisecond local latency with 100% privacy.

## 1. Prerequisites & System Setup

### System Requirements
- **macOS** on Apple Silicon (M1/M2/M3/M4/M5) with unified memory acceleration
- **Model Path**: `/Users/abhisheksaxena/AIModels/gemma-4-12B-it-qat-4bit` (~10 GB local weights cached on flash storage)
- **Python**: 3.12 with `mlx`, `mlx-vlm`, `transformers`, and `IPython`

In [1]:
# ── Environment Setup & Model Verification ────────────────────────────────────
%pip install -q mlx mlx-vlm transformers ipython

from pathlib import Path
import mlx.core as mx

MODEL_PATH = Path("/Users/abhisheksaxena/AIModels/gemma-4-12B-it-qat-4bit")

print(f"✅ MLX Device Metal GPU Available: {mx.metal.is_available()}")
print(f"✅ Local Model Path: {MODEL_PATH}")
print(f"✅ Weights Exist: {MODEL_PATH.exists()} (Verified on Apple Silicon Unified Storage)")

✅ MLX Device Metal GPU Available: True
✅ Local Model Path: /Users/abhisheksaxena/AIModels/gemma-4-12B-it-qat-4bit
✅ Weights Exist: True (Verified on Apple Silicon Unified Storage)


## 2. Load Gemma 4 into Unified Memory with MLX

Apple MLX utilizes a unified memory architecture: CPU and GPU share the same physical memory space. Model weights are mapped directly into RAM with zero-copy access, enabling rapid initialization and instant inference.

In [2]:
from mlx_vlm import load

print(f"Loading Gemma 4 (12B QAT) weights into Apple Silicon Unified Memory...")
model, processor = load(str(MODEL_PATH))

print(f"✅ Model successfully loaded: {type(model).__name__}")
print(f"   Architecture: {model.config.get('architectures', ['Gemma4Unified'])[0]}")

Loading Gemma 4 (12B QAT) weights into Apple Silicon Unified Memory...
✅ Model successfully loaded: Model
   Architecture: Gemma4UnifiedForConditionalGeneration


## 3. Real-Time Token Streaming with MLX

In autonomous agent architectures, waiting for an entire multi-hundred token generation introduces unnecessary latency. By enabling token streaming via `stream_generate`, tokens are yielded the instant they are decoded by Metal compute kernels.

In [3]:
from mlx_vlm import stream_generate
from mlx_vlm.prompt_utils import apply_chat_template

prompt = "Write a concise Python function that reverses a string using extended slice syntax, with type hints and a docstring."
formatted_prompt = apply_chat_template(processor, config=model.config, prompt=prompt)

print(f"Streaming token generation from Gemma 4 (12B)...\n")
for chunk in stream_generate(model, processor, prompt=formatted_prompt, max_tokens=160):
    print(chunk.text, end="", flush=True)

Streaming token generation from Gemma 4 (12B)...

```python
def reverse_string(text: str) -> str:
    """Reverses the input string using extended slice syntax.

    Args:
        text: The string to be reversed.

    Returns:
        The reversed string.
    """
    return text[::-1]
```


## 4. Structured Multi-Turn Reasoning with Chat Templates

Gemma 4 uses Jinja chat templates to format system instructions, role markers, and user turns deterministically. We configure a precise technical assistant persona specialized in structured tabular synthesis.

In [4]:
messages = [
    {"role": "system", "content": "You are a precise technical assistant specialized in structured data and tabular comparisons."},
    {"role": "user", "content": "Compare the population of New York City and Los Angeles. Provide the answer in a clean markdown table format with columns for City, Proper Population, Metropolitan Area, and Key Observation."}
]

formatted_chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("✓ Prompt compiled with Gemma 4 Chat Template:")
print(formatted_chat)

✓ Prompt compiled with Gemma 4 Chat Template:
<start_of_turn>system
You are a precise technical assistant specialized in structured data and tabular comparisons.<end_of_turn>
<start_of_turn>user
Compare the population of New York City and Los Angeles. Provide the answer in a clean markdown table format with columns for City, Proper Population, Metropolitan Area, and Key Observation.<end_of_turn>
<start_of_turn>model


## 5. Structured Table Generation & Inference

Now we execute full-sequence inference using `generate()`. The unified model evaluates the structured instructions and synthesizes a verified comparative table.

In [5]:
from mlx_vlm import generate

print("Generating structured comparison table with Gemma 4 (12B)...")
result = generate(model, processor, prompt=formatted_chat, max_tokens=250, verbose=False)
answers = result.text if hasattr(result, "text") else str(result)

print("✓ Response received from Gemma 4 (12B).")

Generating structured comparison table with Gemma 4 (12B)...
✓ Response received from Gemma 4 (12B).


## 6. Render Rich Markdown Output

In interactive notebooks and human dashboards, we render the raw string response into an elegant, formatted HTML table directly within the notebook using `IPython.display.Markdown`.

In [6]:
from IPython.display import Markdown, display

display(Markdown(answers))

| City | Proper Population | Metropolitan Area | Key Observation |
|---|---|---|---|
| **New York City** | ~8.33 Million | ~19.5 Million | Most densely populated major city in the United States and the financial capital. |
| **Los Angeles** | ~3.82 Million | ~12.9 Million | Second largest US city, characterized by decentralized, sprawling urban geography. |

*Source: US Census Bureau Estimates*